# Visualize abundant TCR sequences

Download the best matching structures and visualize the clonotype residues in vdW representation while the rest of the protein is shown as ribbons.

In [ ]:
from pathlib import Path

import pandas as pd
import requests
from Bio.Data.IUPACData import protein_letters_3to1_extended
from Bio.PDB import PDBParser
from IPython.display import HTML, display

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "TCRTools").exists() and not (NOTEBOOK_DIR / "data").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR / "TCRTools"

OUTPUT_DIR = NOTEBOOK_DIR / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
BEST_HITS_TABLE = OUTPUT_DIR / "best_tcr_structure_hits.csv"
best_hits = pd.read_csv(BEST_HITS_TABLE)
best_hits[["tcr_unit_name", "chain_type", "identifier", "auth_asym_ids", "highlight_sequence"]]

In [ ]:
def clean_sequence(sequence):
    return "".join(str(sequence).upper().replace("-", "").split())


def download_pdb(pdb_id, output_dir):
    pdb_id = str(pdb_id).upper()
    output_path = Path(output_dir) / f"{pdb_id}.pdb"
    if not output_path.exists():
        response = requests.get(f"https://files.rcsb.org/download/{pdb_id}.pdb", timeout=30)
        response.raise_for_status()
        output_path.write_text(response.text)
    return output_path


def first_auth_chain(auth_asym_ids):
    if pd.isna(auth_asym_ids):
        return ""
    return str(auth_asym_ids).split(",")[0].strip()


def residues_for_chain(pdb_path, chain_id):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(Path(pdb_path).stem, pdb_path)
    chain = structure[0][chain_id]
    residues = []
    for residue in chain:
        if residue.id[0] != " ":
            continue
        residue_number = str(residue.id[1])
        insertion_code = residue.id[2].strip()
        residues.append(
            {
                "residue_number": residue_number,
                "insertion_code": insertion_code,
                "residue_name": residue.resname,
                "one_letter": protein_letters_3to1_extended.get(residue.resname.title(), "X"),
            }
        )
    return residues


def locate_motif_residues(pdb_path, chain_id, motif_sequence):
    motif = clean_sequence(motif_sequence)
    if not motif:
        return []
    residues = residues_for_chain(pdb_path, chain_id)
    sequence = "".join(residue["one_letter"] for residue in residues)
    start = sequence.find(motif)
    if start == -1:
        return []
    return [residue["residue_number"] for residue in residues[start : start + len(motif)]]


def residue_numbers_to_range(residue_numbers):
    numbers = [int(value) for value in residue_numbers]
    if not numbers:
        return ""
    ranges = []
    start = previous = numbers[0]
    for number in numbers[1:]:
        if number == previous + 1:
            previous = number
            continue
        ranges.append(f"{start}-{previous}" if start != previous else str(start))
        start = previous = number
    ranges.append(f"{start}-{previous}" if start != previous else str(start))
    return "+".join(ranges)


def ngl_selection(chain_id, residue_numbers=None):
    if residue_numbers:
        residue_expression = residue_numbers_to_range(residue_numbers).replace("+", " or ")
        return f":{chain_id} and ({residue_expression})"
    return f":{chain_id}"


def summarize_pdb_chains(pdb_path):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(Path(pdb_path).stem, pdb_path)
    rows = []
    for chain in structure[0]:
        residues = [residue for residue in chain if residue.id[0] == " "]
        atoms = sum(1 for residue in residues for _atom in residue)
        rows.append({"chain": chain.id, "atoms": atoms, "residues": len(residues)})
    return pd.DataFrame(rows)


def chain_label_positions(pdb_path, chain_ids=None):
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure(Path(pdb_path).stem, pdb_path)
    requested = set(chain_ids) if chain_ids is not None else None
    labels = []
    for chain in structure[0]:
        if requested is not None and chain.id not in requested:
            continue
        residues = [residue for residue in chain if residue.id[0] == " "]
        if not residues:
            continue
        residue = residues[len(residues) // 2]
        labels.append({"chain": chain.id, "residue_number": str(residue.id[1])})
    return labels


def add_chain_labels(view, pdb_path, chain_ids=None):
    for label in chain_label_positions(pdb_path, chain_ids=chain_ids):
        selection = f":{label['chain']} and {label['residue_number']} and .CA"
        view.add_label(
            selection=selection,
            labelType="text",
            labelText=f"chain {label['chain']}",
            color="black",
            labelSize=2.0,
            zOffset=2.0,
        )
    return view

In [ ]:
visualization_targets = []

for _, hit in best_hits.iterrows():
    structure_path = download_pdb(hit["entry_id"], OUTPUT_DIR)
    chain_id = first_auth_chain(hit["auth_asym_ids"])
    residue_numbers = locate_motif_residues(structure_path, chain_id, hit.get("highlight_sequence", ""))
    visualization_targets.append(
        {
            **hit.to_dict(),
            "structure_path": structure_path.as_posix(),
            "display_chain": chain_id,
            "highlight_residues": ";".join(residue_numbers),
            "highlight_range": residue_numbers_to_range(residue_numbers),
        }
    )

visualization_targets = pd.DataFrame(visualization_targets)
visualization_targets.to_csv(OUTPUT_DIR / "resolved_visualization_targets.csv", index=False)
visualization_targets[["tcr_unit_name", "chain_type", "entry_id", "display_chain", "highlight_label", "highlight_range"]]

In [ ]:
for path in visualization_targets["structure_path"].drop_duplicates():
    display(HTML(f"<h4>{Path(path).name}</h4>"))
    display(summarize_pdb_chains(path))

In [ ]:
import nglview as nv

for _, target in visualization_targets.iterrows():
    display(HTML(f"<h3>{target['tcr_unit_name']} ({target['chain_type']}) - {target['identifier']}</h3>"))

    view = nv.show_structure_file(target["structure_path"])
    view.clear_representations()
    view.add_cartoon(selection="protein", color="lightgray")
    view.add_cartoon(selection=ngl_selection(target["display_chain"]), color="royalblue")
    add_chain_labels(view, target["structure_path"])

    residue_numbers = [residue for residue in str(target["highlight_residues"]).split(";") if residue]
    if residue_numbers:
        view.add_spacefill(selection=ngl_selection(target["display_chain"], residue_numbers), color="orange", scale=0.45)

    view.center(ngl_selection(target["display_chain"]))
    display(view)